In [0]:
-- Step 1 - Create satellite type table in silver layer
-- Changes process by SCD2.
-- Actual row is row with value equal '9999-12-31 23:59:59' in effective_to column.

-- drop table nleshin_catalog.silver_layer.objects_description_satellite purge;

create table if not exists nleshin_catalog.silver_layer.objects_description_satellite(
    file_surrogate_key string not null,
    length bigint,
    parsed_content variant,
    effective_from timestamp,
    effective_to timestamp not null,
    hash_diff string,
    sys_inserted_stamp timestamp not null default current_timestamp(),
    sys_updated_stamp timestamp,
    job_run_id_inserted string,
    job_run_id_updated string,
    primary key (file_surrogate_key, effective_from)
)
CLUSTER BY AUTO
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
-- Step 2 - Create satellite type table in silver layer

with src as (
    select 
        null as key_for_merge_with_trg,
        src.*
    from nleshin_catalog.stg_layer.objects_description src
    left join nleshin_catalog.silver_layer.objects_description_satellite trg
        on src.file_name = trg.file_surrogate_key
        and src.hash_diff != trg.hash_diff
        and trg.effective_to = '9999-12-31 23:59:59'
    where trg.file_surrogate_key is not null
    union all
    select 
        src.file_surrogate_key as key_for_merge_with_trg,
        src.*
    from nleshin_catalog.stg_layer.objects_description src
)
merge into nleshin_catalog.silver_layer.objects_description_satellite trg
using src
    on src.key_for_merge_with_trg = trg.file_surrogate_key 
    and src.hash_diff != trg.hash_diff 
    and trg.effective_to = '9999-12-31 23:59:59'
when matched then update set
    effective_to = dateadd(SECOND, -cast(1 as int), src.src_inserted_stamp),
    sys_updated_stamp = current_timestamp(),
    job_run_id_updated = :job_run_id
when not matched then insert (
    file_surrogate_key,
    length,
    parsed_content,
    effective_from,
    effective_to,
    hash_diff,
    job_run_id_inserted
) values (
    src.file_surrogate_key,
    src.length,
    src.parsed_content,
    src.src_inserted_stamp,
    to_timestamp('9999-12-31 23:59:59'),
    src.hash_diff,
    :job_run_id
);